# Data Analysis Module

Performs calculations and generates statistics for emission trend visualization.

**Capabilities:**
- Calculate trend statistics (YoY changes, linear regression)
- Country summary statistics
- Multi-country comparison data
- Carbon intensity calculations
- Regional averages
- **Period-based analysis** (Pre/Post Arab Spring, Zohr, COVID, IMF)
- **CAGR calculation** (Compound Annual Growth Rate)
- **Peak/Valley detection** (significant years)
- **GDP per capita** derivation
- **Consumption vs Production** gap analysis

In [ ]:
# Import required libraries
import os
import sys
import pandas as pd
import numpy as np
from typing import Dict, List, Optional, Tuple

# Import from data module
import sys
sys.path.append('..')

from data.fetch_data import DataFetcher, COUNTRY_CODES, ISO_TO_COUNTRY

ModuleNotFoundError: No module named 'data.fetch_data'

## EmissionAnalyzer Class

In [ ]:
class EmissionAnalyzer:
    """Analyzer class for emission data"""

    def __init__(self, data_fetcher: Optional[DataFetcher] = None):
        self.fetcher = data_fetcher or DataFetcher(use_cache=True)
        self.fetcher.fetch_owid_data()

    def calculate_trend_stats(self, country_code: str, metric: str = "co2") -> Dict:
        """Calculate trend statistics for a country"""
        data = self.fetcher.get_country_data_owid(country_code)

        if data.empty:
            return {}

        values = data[metric].fillna(0).values
        years = data["year"].values

        # Calculate year-over-year changes
        yoy_changes = np.diff(values) / values[:-1] * 100
        yoy_changes = np.insert(yoy_changes, 0, 0)

        # Calculate trend line (simple linear regression)
        if len(values) > 1:
            x = np.arange(len(values))
            coeffs = np.polyfit(x, values, 1)
            slope = coeffs[0]
            trend_direction = "increasing" if slope > 0 else "decreasing"
        else:
            slope = 0
            trend_direction = "stable"

        # Overall change
        if len(values) >= 2:
            overall_change = ((values[-1] - values[0]) / values[0] * 100) if values[0] != 0 else 0
        else:
            overall_change = 0

        return {
            "metric": metric,
            "start_value": float(values[0]) if len(values) > 0 else 0,
            "end_value": float(values[-1]) if len(values) > 0 else 0,
            "min_value": float(values.min()) if len(values) > 0 else 0,
            "max_value": float(values.max()) if len(values) > 0 else 0,
            "mean_value": float(values.mean()) if len(values) > 0 else 0,
            "overall_change_pct": float(overall_change),
            "trend_direction": trend_direction,
            "trend_slope": float(slope),
            "yoy_changes": [float(x) for x in yoy_changes],
            "years": [int(y) for y in years],
            "values": [float(v) for v in values],
        }

## Country Summary

In [ ]:
    def get_country_summary(self, country_code: str, year: int = None) -> Dict:
        """Get summary statistics for a country"""
        data = self.fetcher.get_country_data_owid(country_code)

        if data.empty:
            return {}

        if year is None:
            data = data.dropna(subset=["co2", "co2_per_capita"], how="all")
            if data.empty:
                return {}
            year = int(data["year"].max())

        row = data[data["year"] == year]
        if row.empty:
            row = data.iloc[-1:]
            year = int(row["year"].values[0])

        if row.empty:
            return {}

        r = row.iloc[0]
        global_stats = self.fetcher.get_global_stats(year)

        summary = {
            "country_code": country_code,
            "country_name": ISO_TO_COUNTRY.get(country_code, country_code),
            "year": year,
            "co2_total": float(r["co2"]) if pd.notna(r.get("co2")) else None,
            "co2_per_capita": float(r["co2_per_capita"]) if pd.notna(r.get("co2_per_capita")) else None,
            "share_global_co2": float(r["share_global_co2"]) if pd.notna(r.get("share_global_co2")) else None,
            "gdp": float(r["gdp"]) if pd.notna(r.get("gdp")) else None,
            "population": float(r["population"]) if pd.notna(r.get("population")) else None,
            "global_co2_total": float(global_stats["total_co2"]) if global_stats else None,
            "global_co2_per_capita": float(global_stats["co2_per_capita"]) if global_stats else None,
        }

        if summary["co2_per_capita"] and global_stats and global_stats["co2_per_capita"]:
            summary["vs_global_per_capita"] = summary["co2_per_capita"] / global_stats["co2_per_capita"]

        return summary

## Comparison Data

In [ ]:
    def get_comparison_data(
        self, primary_country: str, comparison_countries: List[str], year: int = None
    ) -> Dict:
        """Get comparison data for multiple countries"""
        comparison = self.fetcher.get_comparison_data(
            primary_country, comparison_countries, year_range=(2005, 2024)
        )

        if year is None:
            year = 2023

        global_stats = self.fetcher.get_global_stats(year)

        result = {
            "primary_country": primary_country,
            "comparison_countries": comparison_countries,
            "year": year,
            "global_stats": global_stats,
            "countries": [],
        }

        for country_code in [primary_country] + comparison_countries:
            if country_code in comparison:
                data = comparison[country_code]
                latest_idx = len(data["year"]) - 1
                if latest_idx >= 0:
                    country_info = {
                        "code": country_code,
                        "name": ISO_TO_COUNTRY.get(country_code, country_code),
                        "year": data["year"][latest_idx],
                        "co2": data["co2"][latest_idx],
                        "co2_per_capita": data["co2_per_capita"][latest_idx],
                        "share_global_co2": data["share_global_co2"][latest_idx],
                        "gdp": data["gdp"][latest_idx],
                        "population": data["population"][latest_idx],
                        "all_years": data["year"],
                        "all_co2": data["co2"],
                        "all_co2_per_capita": data["co2_per_capita"],
                    }
                    result["countries"].append(country_info)

        return result

## Carbon Intensity & Regional Averages

In [ ]:
    def calculate_carbon_intensity(self, country_code: str) -> Dict:
        """Calculate carbon intensity (CO2 per unit GDP)"""
        summary = self.get_country_summary(country_code)

        if not summary or not summary.get("co2_total") or not summary.get("gdp"):
            return {}

        carbon_intensity = (
            summary["co2_total"] / (summary["gdp"] / 1_000_000)
            if summary["gdp"] > 0
            else 0
        )

        return {
            "country_code": country_code,
            "year": summary.get("year"),
            "co2_total_mt": summary["co2_total"],
            "gdp_billions": summary["gdp"] / 1_000_000_000,
            "carbon_intensity": carbon_intensity,
            "unit": "metric tons CO2 per million USD GDP",
        }

    def get_regional_averages(self, countries: List[str], year: int = None) -> Dict:
        """Calculate regional averages for a set of countries"""
        totals = {"co2": [], "co2_per_capita": [], "gdp": [], "population": []}

        for code in countries:
            summary = self.get_country_summary(code, year)
            if summary:
                for key in totals.keys():
                    value = summary.get(key)
                    if value:
                        totals[key].append(value)

        result = {}
        for key, values in totals.items():
            if values:
                result[f"{key}_avg"] = np.mean(values)
                result[f"{key}_median"] = np.median(values)
                result[f"{key}_std"] = np.std(values)

        return result

## Enhanced Analysis Methods (Presentation-Ready)

In [ ]:
    def calculate_cagr(self, start_value: float, end_value: float, years: int) -> float:
        """
        Calculate Compound Annual Growth Rate (CAGR)
        CAGR = (End Value / Start Value)^(1/years) - 1
        """
        if start_value <= 0 or years <= 0:
            return None
        return (end_value / start_value) ** (1/years) - 1

    def get_period_analysis(self, country_code: str, period: str = "full") -> Dict:
        """
        Analyze emissions for specific historical periods.
        Critical for understanding events like Arab Spring, Zohr, COVID, IMF.
        """
        period_configs = {
            "full": {"years": (2005, 2023), "name": "Full Period", 
                     "description": "2005-2023: Pre/Post Arab Spring, Zohr, IMF Reform, COVID"},
            "pre_arab_spring": {"years": (2005, 2010), "name": "Pre-Arab Spring",
                              "description": "2005-2010: High GDP growth 5-7%, energy subsidies"},
            "post_arab_spring": {"years": (2011, 2015), "name": "Post-Arab Spring",
                              "description": "2011-2015: Political transition, emissions plateau"},
            "imf_reform": {"years": (2016, 2019), "name": "IMF Reform Period",
                          "description": "2016-2019: EGP floated, devalued 50%, subsidies cut"},
            "post_zohr": {"years": (2018, 2023), "name": "Post-Zohr Gas",
                        "description": "2018-2023: Zohr online 2017, net gas exporter"},
            "covid": {"years": (2019, 2023), "name": "COVID Era",
                     "description": "2019-2023: Pandemic impact, rebound recovery"}
        }
        
        if period not in period_configs:
            period = "full"
        
        config = period_configs[period]
        start_year, end_year = config["years"]
        data = self.fetcher.get_country_data_owid(country_code, (start_year, end_year))
        
        if data.empty:
            return {}
        
        start_row = data[data["year"] == start_year]
        end_row = data[data["year"] == end_year]
        
        if start_row.empty or end_row.empty:
            return {}
        
        start_data = start_row.iloc[0]
        end_data = end_row.iloc[0]
        num_years = end_year - start_year
        
        result = {
            "period": period,
            "period_name": config["name"],
            "description": config["description"],
            "start_year": start_year,
            "end_year": end_year,
            "num_years": num_years,
            "start_co2": float(start_data["co2"]) if pd.notna(start_data.get("co2")) else None,
            "end_co2": float(end_data["co2"]) if pd.notna(end_data.get("co2")) else None,
            "start_co2_per_capita": float(start_data["co2_per_capita"]) if pd.notna(start_data.get("co2_per_capita")) else None,
            "end_co2_per_capita": float(end_data["co2_per_capita"]) if pd.notna(end_data.get("co2_per_capita")) else None,
        }
        
        if result["start_co2"] and result["end_co2"] and result["start_co2"] > 0:
            result["co2_change_abs"] = result["end_co2"] - result["start_co2"]
            result["co2_change_pct"] = ((result["end_co2"] - result["start_co2"]) / result["start_co2"]) * 100
            result["co2_cagr"] = self.calculate_cagr(result["start_co2"], result["end_co2"], num_years)
        
        return result

    def find_peak_years(self, country_code: str, metric: str = "co2") -> Dict:
        """
        Find peak (maximum) and valley (minimum) years.
        Useful for identifying significant events (e.g., 2017 Zohr peak).
        """
        data = self.fetcher.get_country_data_owid(country_code)
        
        if data.empty:
            return {}
        
        values = data[metric].fillna(0).values
        years = data["year"].values
        
        peak_idx = values.argmax()
        valley_idx = values.argmin()
        
        return {
            "metric": metric,
            "peak_year": int(years[peak_idx]),
            "peak_value": float(values[peak_idx]),
            "valley_year": int(years[valley_idx]),
            "valley_value": float(values[valley_idx]),
            "all_years": [int(y) for y in years],
            "all_values": [float(v) for v in values]
        }

    def get_gdp_per_capita(self, country_code: str, year: int = None) -> Optional[float]:
        """
        Calculate GDP per capita = GDP / Population
        Key development indicator for climate justice arguments.
        """
        summary = self.get_country_summary(country_code, year)
        if summary and summary.get("gdp") and summary.get("population") and summary["population"] > 0:
            return summary["gdp"] / summary["population"]
        return None

    def get_consumption_production_gap(self, country_code: str, year: int = None) -> Dict:
        """
        Calculate consumption vs production-based CO2 gap.
        Climate justice: Should Egypt be blamed for imported emissions?
        """
        data = self.fetcher.get_country_data_owid(country_code)
        if year is None:
            year = 2023
        row = data[data["year"] == year]
        if row.empty:
            row = data.iloc[-1:]
        r = row.iloc[0]
        
        production = float(r["co2"]) if pd.notna(r.get("co2")) else None
        consumption = float(r["consumption_co2"]) if pd.notna(r.get("consumption_co2")) else None
        
        result = {
            "country_code": country_code,
            "year": int(r["year"]),
            "production_co2": production,
            "consumption_co2": consumption,
        }
        
        if production and consumption:
            result["gap"] = consumption - production
            result["gap_pct"] = ((consumption - production) / production) * 100 if production > 0 else None
            result["is_importer"] = consumption > production
        
        return result

    def get_emission_sources(self, country_code: str, year: int = None) -> Dict:
        """
        Break down CO2 emissions by source (coal, oil, gas, cement).
        Useful for understanding energy mix and policy.
        """
        data = self.fetcher.get_country_data_owid(country_code)
        if year is None:
            year = 2023
        row = data[data["year"] == year]
        if row.empty:
            row = data.iloc[-1:]
        r = row.iloc[0]
        
        return {
            "country_code": country_code,
            "year": int(r["year"]),
            "total_co2": float(r["co2"]) if pd.notna(r.get("co2")) else None,
            "coal_co2": float(r["coal_co2"]) if pd.notna(r.get("coal_co2")) else 0,
            "oil_co2": float(r["oil_co2"]) if pd.notna(r.get("oil_co2")) else 0,
            "gas_co2": float(r["gas_co2"]) if pd.notna(r.get("gas_co2")) else 0,
            "cement_co2": float(r["cement_co2"]) if pd.notna(r.get("cement_co2")) else 0,
            "flaring_co2": float(r["flaring_co2"]) if pd.notna(r.get("flaring_co2")) else 0,
        }

## Number Formatting

In [ ]:
    def format_number(self, value: float, unit: str = "") -> str:
        """Format a number for display"""
        if value is None:
            return "N/A"

        if unit == "CO2":
            if value >= 1000:
                return f"{value / 1000:.2f} Gt"
            return f"{value:.2f} Mt"

        if unit == "population":
            if value >= 1_000_000_000:
                return f"{value / 1_000_000_000:.2f} B"
            if value >= 1_000_000:
                return f"{value / 1_000_000:.2f} M"
            if value >= 1_000:
                return f"{value / 1_000:.1f} K"
            return f"{value:.0f}"

        if unit == "GDP":
            if value >= 1_000_000_000_000:
                return f"${value / 1_000_000_000_000:.2f} T"
            if value >= 1_000_000_000:
                return f"${value / 1_000_000_000:.2f} B"
            if value >= 1_000_000:
                return f"${value / 1_000_000:.2f} M"
            return f"${value:.2f}"

        if unit == "percentage":
            return f"{value:.2f}%"

        if unit == "per_capita":
            return f"{value:.2f} metric tons"

        if abs(value) >= 1_000_000:
            return f"{value / 1_000_000:.2f}M"
        if abs(value) >= 1_000:
            return f"{value / 1_000:.1f}K"
        return f"{value:.2f}"

---

## Test & Execute

In [ ]:
# Initialize the analyzer
analyzer = EmissionAnalyzer()

In [ ]:
# Egypt Summary (2023)
print("=" * 60)
print("Egypt Summary (2023)")
print("=" * 60)

egypt_summary = analyzer.get_country_summary("EGY", 2023)
for key, value in egypt_summary.items():
    print(f"  {key}: {value}")

In [ ]:
# Egypt CO2 Trend Analysis
print("\n" + "=" * 60)
print("Egypt CO2 Trend Analysis (2005-2024)")
print("=" * 60)

trend = analyzer.calculate_trend_stats("EGY", "co2")
print(f"  Start Value (2005): {trend.get('start_value', 0):.2f} Mt")
print(f"  End Value (2024): {trend.get('end_value', 0):.2f} Mt")
print(f"  Min Value: {trend.get('min_value', 0):.2f} Mt")
print(f"  Max Value: {trend.get('max_value', 0):.2f} Mt")
print(f"  Overall Change: {trend.get('overall_change_pct', 0):.2f}%")
print(f"  Trend Direction: {trend.get('trend_direction', 'N/A')}")

In [ ]:
# Per Capita Trend Analysis
print("\n" + "=" * 60)
print("Egypt Per Capita CO2 Trend (2005-2024)")
print("=" * 60)

per_capita = analyzer.calculate_trend_stats("EGY", "co2_per_capita")
print(f"  Start Value (2005): {per_capita.get('start_value', 0):.2f} tonnes")
print(f"  End Value (2024): {per_capita.get('end_value', 0):.2f} tonnes")
print(f"  Mean: {per_capita.get('mean_value', 0):.2f} tonnes")
print(f"  Overall Change: {per_capita.get('overall_change_pct', 0):.2f}%")
print(f"  Trend Direction: {per_capita.get('trend_direction', 'N/A')}")

In [ ]:
# Comparison Data
print("\n" + "=" * 60)
print("Comparison: Egypt, India, Saudi Arabia, World")
print("=" * 60)

comp = analyzer.get_comparison_data("EGY", ["IND", "SAU", "WLD"])

print(f"\n{'Country':<20} {'CO2 (Mt)':<15} {'Per Capita (t)':<18} {'Share (%)':<12}")
print("-" * 65)
for country in comp["countries"]:
    print(f"{country['name']:<20} {country.get('co2', 0) or 0:<15.2f} {country.get('co2_per_capita', 0) or 0:<18.2f} {country.get('share_global_co2', 0) or 0:<12.3f}")

In [ ]:
# Carbon Intensity Analysis
print("\n" + "=" * 60)
print("Carbon Intensity Analysis")
print("=" * 60)

countries_to_analyze = ["EGY", "IND", "SAU", "USA", "CHN"]

print(f"\n{'Country':<20} {'CO2 (Mt)':<12} {'GDP (B$)':<15} {'Intensity':<20}")
print("-" * 67)
for code in countries_to_analyze:
    intensity = analyzer.calculate_carbon_intensity(code)
    if intensity:
        name = ISO_TO_COUNTRY.get(code, code)
        print(f"{name:<20} {intensity['co2_total_mt']:<12.2f} ${intensity['gdp_billions']:<14.2f} {intensity['carbon_intensity']:<20.2f}")

---

## Enhanced Analysis Tests (Presentation-Ready)

In [ ]:
# Period-Based Analysis (Critical for Historical Events)
print("=" * 70)
print("PERIOD-BASED ANALYSIS - Egypt CO2 Emissions")
print("=" * 70)

periods = ["full", "pre_arab_spring", "post_arab_spring", "imf_reform", "post_zohr", "covid"]

print(f"\n{'Period':<20} {'Start CO2':<12} {'End CO2':<12} {'Change %':<12} {'CAGR':<10}")
print("-" * 66)

for period in periods:
    result = analyzer.get_period_analysis("EGY", period)
    if result:
        name = result.get('period_name', period)
        start = result.get('start_co2', 0) or 0
        end = result.get('end_co2', 0) or 0
        change = result.get('co2_change_pct', 0) or 0
        cagr = result.get('co2_cagr', 0) or 0
        if cagr:
            print(f"{name:<20} {start:<12.1f} {end:<12.1f} {change:>+10.1f}% {cagr:>+8.2%}")
        else:
            print(f"{name:<20} {start:<12.1f} {end:<12.1f} {change:>+10.1f}% {'N/A':<10}")

In [ ]:
# Peak Year Detection (Important for Annotations)
print("\n" + "=" * 70)
print("PEAK AND VALLEY ANALYSIS - Egypt")
print("=" * 70)

peaks = analyzer.find_peak_years("EGY", "co2")
if peaks:
    print(f"\nPeak Year: {peaks['peak_year']} - Value: {peaks['peak_value']:.2f} Mt")
    print(f"Valley Year: {peaks['valley_year']} - Value: {peaks['valley_value']:.2f} Mt")
    print(f"\nNote: 2017 peak corresponds to Zohr gas field discovery")

In [ ]:
# GDP per Capita (Key Development Indicator)
print("\n" + "=" * 70)
print("GDP PER CAPITA ANALYSIS")
print("=" * 70)

countries_gdp = ["EGY", "IND", "CHN", "USA", "DEU"]
print(f"\n{'Country':<20} {'GDP per Capita (USD)':<25} {'CO2 per Capita (t)':<20}")
print("-" * 65)
for code in countries_gdp:
    gdp_pc = analyzer.get_gdp_per_capita(code, 2023)
    summary = analyzer.get_country_summary(code, 2023)
    co2_pc = summary.get('co2_per_capita', 0) if summary else 0
    name = ISO_TO_COUNTRY.get(code, code)
    if gdp_pc:
        print(f"{name:<20} ${gdp_pc:<24,.0f} {co2_pc:<20.2f}")
    else:
        print(f"{name:<20} {'N/A':<25} {co2_pc:<20.2f}")

In [ ]:
# Consumption vs Production Gap (Climate Justice)
print("\n" + "=" * 70)
print("CONSUMPTION VS PRODUCTION CO2 GAP (Climate Justice)")
print("=" * 70)
print("\nPositive gap = Net importer of emissions")
print("Negative gap = Net exporter of emissions")

countries_justice = ["EGY", "IND", "CHN", "USA"]
print(f"\n{'Country':<15} {'Production':<12} {'Consumption':<12} {'Gap':<12} {'Importer?':<10}")
print("-" * 60)
for code in countries_justice:
    gap = analyzer.get_consumption_production_gap(code, 2023)
    if gap and gap.get('production_co2'):
        name = ISO_TO_COUNTRY.get(code, code)
        prod = gap.get('production_co2', 0) or 0
        cons = gap.get('consumption_co2', 0) or 0
        g = gap.get('gap', 0) or 0
        is_imp = "Yes" if gap.get('is_importer') else "No"
        print(f"{name:<15} {prod:<12.2f} {cons:<12.2f} {g:>+12.2f} {is_imp:<10}")

In [ ]:
# Emission Sources Breakdown
print("\n" + "=" * 70)
print("EMISSION SOURCES BREAKDOWN - Egypt 2023")
print("=" * 70)

sources = analyzer.get_emission_sources("EGY", 2023)
if sources:
    total = sources.get('total_co2', 0) or 0
    print(f"\nTotal CO2: {total:.2f} Mt")
    print(f"\nBy Source:")
    print(f"  Coal:    {sources.get('coal_co2', 0):.2f} Mt")
    print(f"  Oil:     {sources.get('oil_co2', 0):.2f} Mt")
    print(f"  Gas:     {sources.get('gas_co2', 0):.2f} Mt")
    print(f"  Cement:  {sources.get('cement_co2', 0):.2f} Mt")
    print(f"  Flaring: {sources.get('flaring_co2', 0):.2f} Mt")
    
    # Calculate percentages
    if total > 0:
        print(f"\nPercentage breakdown:")
        print(f"  Gas:     {sources.get('gas_co2', 0)/total*100:.1f}%")
        print(f"  Oil:     {sources.get('oil_co2', 0)/total*100:.1f}%")

---

**Next:** Continue to [Countries Notebook](countries.ipynb) for country configurations.